# 01 - Data Loading
Load CMEMS oceanographic data and AIS fishing effort (2015-2024)

In [ ]:
!pip install -q xarray netCDF4 pandas numpy matplotlib

In [ ]:
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/fishing_project/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load oceanographic data
print("Loading CMEMS Global Ocean Physics Reanalysis...")
physics_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_phy_my_0.083deg_P1M-m_1777301394294.nc")
print(f"Variables: {list(physics_ds.data_vars)}")
print(f"Shape: {physics_ds.dims}")

print("\nLoading CMEMS Global Ocean Biogeochemistry Hindcast...")
bgc_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_bgc_my_0.25deg_P1M-m_1777301388002.nc")
print(f"Variables: {list(bgc_ds.data_vars)}")
print(f"Shape: {bgc_ds.dims}")

Loading CMEMS Global Ocean Physics Reanalysis...
Variables: ['thetao', 'uo', 'vo', 'zos']
Shape: FrozenMappingWarningOnValuesAccess({'time': 120, 'depth': 5, 'latitude': 121, 'longitude': 72})

Loading CMEMS Global Ocean Biogeochemistry Hindcast...
Variables: ['chl', 'nppv']
Shape: FrozenMappingWarningOnValuesAccess({'time': 120, 'depth': 5, 'latitude': 41, 'longitude': 25})


In [ ]:
# Load AIS fishing effort data (2015-2024)
print("Loading AIS fishing effort CSVs...")
ais_files = [f"ais_effort_{year}.csv" for year in range(2020, 2025)]
ais_dfs = [pd.read_csv(DATA_DIR + f) for f in ais_files]
ais_df = pd.concat(ais_dfs, ignore_index=True)

print(f"AIS records: {len(ais_df):,}")
print(f"Date range: {ais_df['date'].min()} to {ais_df['date'].max()}")
print(f"Columns: {list(ais_df.columns)}")

In [ ]:
# Time range verification
print("\n=== Time Range Verification ===")
print(f"Physics data: {physics_ds.time.min().values} to {physics_ds.time.max().values}")
print(f"BGC data: {bgc_ds.time.min().values} to {bgc_ds.time.max().values}")
# print(f"AIS data: {ais_df['date'].min()} to {ais_df['date'].max()}")


=== Time Range Verification ===
Physics data: 2015-01-01T00:00:00.000000000 to 2024-12-01T00:00:00.000000000
BGC data: 2015-01-01T00:00:00.000000000 to 2024-12-01T00:00:00.000000000


In [ ]:
# Save summary
summary = {
    'physics_shape': dict(physics_ds.dims),
    'bgc_shape': dict(bgc_ds.dims),
    # 'ais_records': len(ais_df),
    'date_range': f"2014-01-01 to 2024-12-31"
}

import json
with open(DATA_DIR + 'data_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✔ Data loading complete")


✅ Data loading complete


/tmp/ipykernel_1177/1982751055.py:3: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'physics_shape': dict(physics_ds.dims),
/tmp/ipykernel_1177/1982751055.py:4: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'bgc_shape': dict(bgc_ds.dims),


# 02 - Preprocessing


In [ ]:
"""
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np
"""
from scipy.interpolate import interp1d
"""
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/fishing_project/""""

In [ ]:
# West Philippine Sea bounding box
LAT_MIN, LAT_MAX = 10, 20
LON_MIN, LON_MAX = 114, 120
DATE_START = "2014-01-01"
DATE_END = "2024-12-31"

## STEP 1: Load and Subset Datasets

In [ ]:
# Physics (0.083° resolution)
physics_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_phy_my_0.083deg_P1M-m_1777301394294.nc")

print(f"Available physics depths: {physics_ds.depth.values[:5]}")

# Select depth first with method='nearest', then slice spatial/temporal
physics_subset = physics_ds.sel(depth=0.49, method='nearest').sel(
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX),
    time=slice(DATE_START, DATE_END)
)

# BGC (0.25° resolution - this is our TARGET grid)
bgc_ds = xr.open_dataset(DATA_DIR + "cmems_mod_glo_bgc_my_0.25deg_P1M-m_1777301388002.nc")

print(f"Available BGC depths: {bgc_ds.depth.values[:5]}")

bgc_subset = bgc_ds.sel(
    depth=slice(0.51, 5.14),
    latitude=slice(LAT_MIN, LAT_MAX),
    longitude=slice(LON_MIN, LON_MAX),
    time=slice(DATE_START, DATE_END)
).mean(dim='depth')

print(f"\nPhysics (0.083°): {physics_subset.dims}")
print(f"BGC (0.25°): {bgc_subset.dims}")

## STEP 2: Monthly Aggregation

In [ ]:
# Monthly aggregation (if not already monthly)
physics_monthly = physics_subset.resample(time='1M').mean()
bgc_monthly = bgc_subset.resample(time='1M').mean()

print(f"Monthly physics: {physics_monthly.dims}")
print(f"Monthly BGC: {bgc_monthly.dims}")

## STEP 3: Define Target Grid and Regrid

In [ ]:
# Use BGC's native coordinates as the standard 0.25° grid
target_lats = bgc_monthly.latitude.values
target_lons = bgc_monthly.longitude.values

print(f"Target grid: {len(target_lats)} × {len(target_lons)} at 0.25° resolution")

# Regrid physics to match BGC grid
physics_regrid = physics_monthly.interp(
    latitude=target_lats,
    longitude=target_lons,
    method='linear'
)

print(f"Physics regridded: {physics_regrid.dims}")

## STEP 4: Verify Alignment

In [ ]:
assert np.allclose(physics_regrid.latitude, bgc_monthly.latitude), "Latitude mismatch!"
assert np.allclose(physics_regrid.longitude, bgc_monthly.longitude), "Longitude mismatch!"
assert len(physics_regrid.time) == len(bgc_monthly.time), "Time mismatch!"

print("✔ All grids aligned to 0.25° resolution")

## STEP 5: Gap Filling

In [ ]:
def fill_gaps(data):
    """Fill NaN gaps with linear interpolation"""
    filled = data.interpolate_na(dim='time', method='linear', fill_value='extrapolate')
    return filled

physics_filled = fill_gaps(physics_regrid)
bgc_filled = fill_gaps(bgc_monthly)

print("✔ Gap filling complete")

## STEP 6: Extract and Normalize Variables

In [ ]:
def normalize(data, method='minmax'):
    """Normalize data using min-max or z-score"""
    if method == 'minmax':
        return (data - data.min()) / (data.max() - data.min())
    elif method == 'zscore':
        return (data - data.mean()) / data.std()
    return data

# Extract variables from correct datasets
# From BGC dataset (biogeochemical)
chl  = normalize(bgc_filled['chl'])   # Chlorophyll-a
nppv = normalize(bgc_filled['nppv'])  # Net primary production

# From Physics dataset
ssh = normalize(physics_filled['zos'])     # Sea surface height
sst = normalize(physics_filled['thetao'])  # Sea surface temperature
uo  = normalize(physics_filled['uo'])      # Eastward sea water velocity
vo  = normalize(physics_filled['vo'])      # Northward sea water velocity

print("✔ Normalization complete")
print(f"  Chl: {chl.shape}")
print(f"  NPP: {nppv.shape}")
print(f"  SSH: {ssh.shape}")
print(f"  SST: {sst.shape}")
print(f"  UO:  {uo.shape}")
print(f"  VO:  {vo.shape}")

## STEP 7: Process AIS Data (Optional)

In [ ]:
ais_files = [f"ais_effort_{year}.csv" for year in range(2020, 2025)]
ais_df = pd.concat([pd.read_csv(DATA_DIR + f) for f in ais_files], ignore_index=True)

# Process AIS to grid
ais_df['lat_bin'] = pd.cut(ais_df['lat'], bins=target_lats, labels=target_lats[:-1])
ais_df['lon_bin'] = pd.cut(ais_df['lon'], bins=target_lons, labels=target_lons[:-1])
ais_df['month'] = pd.to_datetime(ais_df['date']).dt.to_period('M')

# Aggregate fishing effort
ais_grid = ais_df.groupby(['month', 'lat_bin', 'lon_bin'])['fishing_hours'].sum().reset_index()

print(f"AIS grid shape: {ais_grid.shape}")
ais_grid.to_csv(DATA_DIR + 'ais_gridded.csv', index=False)

In [ ]:
# Combine all features into one dataset
preprocessed = xr.Dataset({
    'chl':  chl,
    'nppv': nppv,
    'ssh':  ssh,
    'sst':  sst,
    'uo':   uo,   # Eastward sea water velocity
    'vo':   vo,   # Northward sea water velocity
})

# Save to NetCDF
preprocessed.to_netcdf(DATA_DIR + 'preprocessed_features.nc')

print("\n✔ Preprocessed data saved to preprocessed_features.nc")
print(f"\nFinal dataset dimensions: {preprocessed.dims}")
print(f"Variables: {list(preprocessed.data_vars)}")

# 03 - Model Training
ConvLSTM for spatiotemporal fishing ground prediction

In [ ]:
!pip install -q tensorflow

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Conv2D, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Load preprocessed data
features = xr.open_dataset(DATA_DIR + 'preprocessed_features.nc')
# ais_grid = pd.read_csv(DATA_DIR + 'ais_gridded.csv')

print(f"Features: {features.dims}")
# print(f"AIS: {ais_grid.shape}")

In [ ]:
# Create sequences (3 months input → 1 month output)
SEQ_LEN = 3
PRED_LEN = 1

# Stack features from merged dataset: (time, lat, lon, channels)
# Physics variables (from 0.083° grid, regridded to 0.25°)
X_stack = np.stack([
    features['sst'].values,          # Sea temperature
    features['ssh'].values,          # Sea surface height
    features['vo'].values,           # Eastward sea velocity
    features['uo'].values,           # Northward sea velocity
    features['chl'].values,          # Chlorophyll (BGC)
    features['nppv'].values,         # Net primary production (BGC)
], axis=-1)

# Convert AIS to grid array
n_months = len(features.time)
n_lat = len(features.latitude)
n_lon = len(features.longitude)

y_array = np.zeros((n_months, n_lat, n_lon))
# Fill with AIS data (implementation depends on your AIS grid structure)
# Example: y_array = ais_gridded_data.values if you have it

print(f"X shape: {X_stack.shape}")  # Should be (n_months, n_lat, n_lon, 10)
print(f"y shape: {y_array.shape}")  # Should be (n_months, n_lat, n_lon)

In [ ]:
# Generate sequences
def create_sequences(X, y, seq_len, pred_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len:i+seq_len+pred_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_stack, y_array, SEQ_LEN, PRED_LEN)
print(f"Sequences: {X_seq.shape} → {y_seq.shape}")

In [ ]:
# Train/val/test split (70/15/15)
train_size = int(0.7 * len(X_seq))
val_size = int(0.15 * len(X_seq))

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_val = X_seq[train_size:train_size+val_size]
y_val = y_seq[train_size:train_size+val_size]
X_test = X_seq[train_size+val_size:]
y_test = y_seq[train_size+val_size:]

print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

In [ ]:
# Build ConvLSTM model
model = Sequential([
    ConvLSTM2D(
        filters=64,
        kernel_size=(3, 3),
        padding='same',
        return_sequences=True,
        input_shape=(SEQ_LEN, n_lat, n_lon, 6)
    ),
    BatchNormalization(),
    Dropout(0.2),

    ConvLSTM2D(
        filters=32,
        kernel_size=(3, 3),
        padding='same',
        return_sequences=False
    ),
    BatchNormalization(),
    Dropout(0.2),

    Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['mae']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(DATA_DIR + 'best_model.h5', monitor='val_loss', save_best_only=True)
]

In [ ]:
# Reshape y data to match model output: (samples, lat, lon, 1)
y_train = y_train.squeeze(axis=1)[..., np.newaxis]  # (samples, 1, lat, lon) → (samples, lat, lon, 1)
y_val = y_val.squeeze(axis=1)[..., np.newaxis]
y_test = y_test.squeeze(axis=1)[..., np.newaxis]

print(f"Reshaped y_train: {y_train.shape}")
print(f"Reshaped y_val: {y_val.shape}")
print(f"Reshaped y_test: {y_test.shape}")

# Train model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=8,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Save final model
model.save(DATA_DIR + 'convlstm_model.h5')
# Save training history
import json
with open(DATA_DIR + 'training_history.json', 'w') as f:
    json.dump(history.history, f)
# Save test data for evaluation notebook
np.save(DATA_DIR + 'X_test.npy', X_test)
np.save(DATA_DIR + 'y_test.npy', y_test)
print("\n✔ Model training complete")

# 04 - Model Evaluation
RMSE, MAE, F1, SSI, Wasserstein Distance

In [ ]:
!pip install -q scikit-learn scipy

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from scipy.stats import wasserstein_distance

# Load model and test data
model = tf.keras.models.load_model(DATA_DIR + 'convlstm_model.h5')

# Load test data (saved from 03_model_training.ipynb)
X_test = np.load(DATA_DIR + 'X_test.npy')
y_test = np.load(DATA_DIR + 'y_test.npy')

print(f"Test data: {X_test.shape} → {y_test.shape}")
# Expected: (20, 3, 41, 25, 6) → (20, 41, 25, 1)

In [ ]:
# Generate predictions
# Fill NaN inputs (land/coastal cells) with 0 before predicting
X_test_clean = np.nan_to_num(X_test, nan=0.0)
preds = model.predict(X_test_clean)
print(f"Predictions shape: {preds.shape}")

# Save predictions for 05_visualization.ipynb
np.save(DATA_DIR + 'predictions.npy', preds)

# Flatten and force-clean any remaining NaN → 0
y_flat = np.nan_to_num(y_test.flatten(), nan=0.0)
p_flat = np.nan_to_num(preds.flatten(), nan=0.0)

print(f"NaN in y_flat: {np.isnan(y_flat).sum()}")
print(f"NaN in p_flat: {np.isnan(p_flat).sum()}")



In [ ]:
# 1. RMSE & MAE
rmse = np.sqrt(mean_squared_error(y_flat, p_flat))
mae  = mean_absolute_error(y_flat, p_flat)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")


In [ ]:
# 2. F1 Score (binarized at threshold 0.5)
y_binary    = (y_flat > 0.5).astype(int)
pred_binary = (p_flat > 0.5).astype(int)

f1 = f1_score(y_binary, pred_binary, zero_division=0)
print(f"F1 Score: {f1:.4f}")


In [ ]:
# 3. SSI (Structural Similarity Index)
def calculate_ssi(obs, pred):
    """Compute SSI per sample"""
    C1, C2 = 0.01**2, 0.03**2
    mu_obs, mu_pred     = obs.mean(), pred.mean()
    sigma_obs, sigma_pred = obs.std(), pred.std()
    sigma_cross = np.mean((obs - mu_obs) * (pred - mu_pred))

    luminance = (2 * mu_obs * mu_pred + C1) / (mu_obs**2 + mu_pred**2 + C1)
    contrast  = (2 * sigma_obs * sigma_pred + C2) / (sigma_obs**2 + sigma_pred**2 + C2)
    structure = (sigma_cross + C2/2) / (sigma_obs * sigma_pred + C2/2)

    return luminance * contrast * structure

ssi_scores = []
for i in range(len(y_test)):
    yi = y_test[i].flatten()
    pi = preds[i].flatten()
    m  = ~(np.isnan(yi) | np.isnan(pi))
    ssi_scores.append(calculate_ssi(yi[m], pi[m]))

ssi_mean = np.mean(ssi_scores)
ssi_std  = np.std(ssi_scores)
print(f"SSI: {ssi_mean:.4f} ± {ssi_std:.4f}")


In [ ]:
# 4. Wasserstein Distance
wd_scores = []
for i in range(len(y_test)):
    obs_flat  = y_test[i].flatten()
    pred_flat = preds[i].flatten()
    m = ~(np.isnan(obs_flat) | np.isnan(pred_flat))
    obs_flat, pred_flat = obs_flat[m], pred_flat[m]

    # Avoid division by zero if all zeros
    obs_sum  = obs_flat.sum()
    pred_sum = pred_flat.sum()
    if obs_sum == 0 or pred_sum == 0:
        wd_scores.append(np.nan)
        continue

    obs_dist  = obs_flat / obs_sum
    pred_dist = pred_flat / pred_sum

    wd = wasserstein_distance(
        np.arange(len(obs_dist)),
        np.arange(len(pred_dist)),
        obs_dist,
        pred_dist
    )
    wd_scores.append(wd)

wd_scores = [s for s in wd_scores if not np.isnan(s)]
wd_mean = np.mean(wd_scores)
wd_std  = np.std(wd_scores)
print(f"Wasserstein Distance: {wd_mean:.4f} ± {wd_std:.4f}")


In [ ]:
# Create results table
results = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'F1', 'SSI', 'Wasserstein'],
    'Value': [
        f"{rmse:.3e}",
        f"{mae:.3e}",
        f"{f1:.4f}",
        f"{ssi_mean:.4f}",
        f"{wd_mean:.4f}"
    ],
    'Std Dev': [
        'N/A', 'N/A', 'N/A',
        f"{ssi_std:.4f}",
        f"{wd_std:.4f}"
    ]
})

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(results.to_string(index=False))

results.to_csv(DATA_DIR + 'evaluation_results.csv', index=False)
print("\n✔ Evaluation complete")


# 05 - Visualization
Spatial predictions, error maps, temporal trends

In [ ]:
!pip install -q matplotlib cartopy

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Load predictions
y_test = np.load(DATA_DIR + 'y_test.npy')
preds = np.load(DATA_DIR + 'predictions.npy')

# Coordinate system
LAT_MIN, LAT_MAX = 7, 20
LON_MIN, LON_MAX = 114, 120

In [ ]:
# Custom colormap
cmap = LinearSegmentedColormap.from_list(
    'fishing',
    ['#000033', '#0000FF', '#FF00FF', '#FF0000']
)

In [ ]:
# 1. Observed vs Predicted (6 samples)
n_samples = min(6, len(y_test))
fig, axes = plt.subplots(2, n_samples, figsize=(18, 6))

for i in range(n_samples):
    # Observed
    axes[0, i].imshow(
        y_test[i, :, :, 0],
        extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
        cmap=cmap,
        vmin=0, vmax=1,
        aspect='auto',
        origin='lower'
    )
    axes[0, i].set_title(f'Month {i+1}')
    if i == 0:
        axes[0, i].set_ylabel('Observed\nLatitude')

    # Predicted
    axes[1, i].imshow(
        preds[i, :, :, 0],
        extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
        cmap=cmap,
        vmin=0, vmax=1,
        aspect='auto',
        origin='lower'
    )
    if i == 0:
        axes[1, i].set_ylabel('Predicted\nLatitude')
    axes[1, i].set_xlabel('Longitude')

fig.colorbar(
    axes[0, 0].images[0],
    ax=axes,
    orientation='horizontal',
    fraction=0.05,
    label='Fishing Probability'
)

plt.tight_layout()
plt.savefig(DATA_DIR + 'obs_vs_pred.png', dpi=300)
plt.show()

In [ ]:
# 2. Error Analysis
error = preds - y_test

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Mean error
im1 = axes[0].imshow(
    error.mean(axis=0)[:, :, 0],
    extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    cmap='RdBu_r',
    vmin=-0.3, vmax=0.3,
    aspect='auto',
    origin='lower'
)
axes[0].set_title('Mean Error')
axes[0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0], label='Pred - Obs')

# MAE
im2 = axes[1].imshow(
    np.abs(error).mean(axis=0)[:, :, 0],
    extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    cmap='Reds',
    vmin=0, vmax=0.3,
    aspect='auto',
    origin='lower'
)
axes[1].set_title('MAE')
plt.colorbar(im2, ax=axes[1], label='|Pred - Obs|')

# RMSE per cell
rmse_spatial = np.sqrt((error**2).mean(axis=0)[:, :, 0])
im3 = axes[2].imshow(
    rmse_spatial,
    extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
    cmap='Reds',
    vmin=0, vmax=0.3,
    aspect='auto',
    origin='lower'
)
axes[2].set_title('RMSE per Cell')
plt.colorbar(im3, ax=axes[2], label='RMSE')

for ax in axes:
    ax.set_xlabel('Longitude')

plt.tight_layout()
plt.savefig(DATA_DIR + 'error_maps.png', dpi=300)
plt.show()

In [ ]:
# 3. Temporal Performance
from sklearn.metrics import mean_squared_error

monthly_rmse = []
for i in range(len(y_test)):
    yi = np.nan_to_num(y_test[i].flatten(), nan=0.0)
    pi = np.nan_to_num(preds[i].flatten(), nan=0.0)
    rmse = np.sqrt(mean_squared_error(yi, pi))
    monthly_rmse.append(rmse)

plt.figure(figsize=(10, 4))
plt.plot(monthly_rmse, marker='o', linewidth=2, markersize=6, color='#0077BB')
plt.xlabel('Test Sample (month)')
plt.ylabel('RMSE')
plt.title('Prediction Accuracy Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DATA_DIR + 'temporal_performance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 4. Training History
import json
with open(DATA_DIR + 'training_history.json', 'r') as f:
    history = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss
ax1.plot(history['loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# MAE
ax2.plot(history['mae'], label='Train')
ax2.plot(history['val_mae'], label='Val')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.set_title('Training MAE')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR + 'training_curves.png', dpi=300)
plt.show()

print("\n✅ All visualizations saved")